# Proyecto Oráculo

Análisis y Diseño de Algoritmos

Ustedes solo escriben en la **celda 4**. Todo lo demás ya está hecho.

```
r = oraculo.evaluar(config, instancias, semilla)

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]   ← gratis, sin límite
oraculo.gastado    # cuántos rollouts llevan
```


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*


In [3]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
fatal: destination path 'open-instruct' already exists and is not an empty directory.
LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2


## 2 · El modelo

`cargar_modelo` acepta un alias de la tabla o cualquier id público de Hugging Face (`org/nombre`).

| Alias | Checkpoint | Tamaño | Notas |
|---|---|---|---|
| `pequeno` | `Qwen/Qwen3-1.7B` | 1.7B | el de por defecto; el más rápido |
| `ministral3b` | `mistralai/Ministral-3-3B-Instruct-2512-BF16` | 3.8B | fp16, ~7.7 GB de VRAM |
| `llama3b` | `unsloth/Llama-3.2-3B-Instruct` | 3.2B | fp16, ~6.4 GB de VRAM |
| `qwen8b` | `unsloth/Qwen3-8B-unsloth-bnb-4bit` | 8B | 4-bit; lento en T4 |
| `mistral7b` | `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` | 7B | 4-bit; lento en T4 |

Son tres familias distintas (Qwen, Mistral, Llama): sirve para ver si su configuración generaliza o si solo le funciona a un modelo.

El caché guarda el nombre del modelo en la clave, así que cambiar de modelo **no** reusa respuestas del anterior: vuelve a gastar rollouts.

> Usen el repo `-BF16` de Ministral 3. El repo por defecto es FP8 y la T4 no lo soporta.


In [5]:
from ayudas import cargar_modelo

#  "pequeno"     → Qwen/Qwen3-1.7B
#  "ministral3b" → mistralai/Ministral-3-3B-Instruct-2512-BF16
#  "llama3b"     → unsloth/Llama-3.2-3B-Instruct
#  "qwen8b"      → unsloth/Qwen3-8B-unsloth-bnb-4bit
#  "mistral7b"   → unsloth/mistral-7b-instruct-v0.3-bnb-4bit
modelo = cargar_modelo("pequeno")


cargando pequeno → Qwen/Qwen3-1.7B


RuntimeError: Cannot access accelerator device when none is available.

## 3 · El oráculo

Si Colab se desconecta, descomenten las dos líneas de Drive para no perder el caché.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda)
# oraculo = Oraculo(modelo, busqueda, cache="/content/drive/MyDrive/oraculo_cache.json")

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Aquí escriben ustedes

Abajo hay una búsqueda aleatoria de ejemplo. Bórrenla y pongan su heurística.


In [ ]:
RANURAS = ["rol", "estrategia", "formato", "verificacion"]
MIN_PALABRAS = 6
MAX_PALABRAS = 20
MIN_TEMP = 0.5


def palabras(texto):
    return len(texto.split()) if texto else 0


def parte_valida(texto):
    n = palabras(texto)
    return MIN_PALABRAS <= n <= MAX_PALABRAS


def temp_valida(temperatura):
    return temperatura > MIN_TEMP


def presupuesto_restante():
    return PRESUPUESTO - oraculo.gastado


def puede_evaluar():
    return presupuesto_restante() >= len(INSTANCIAS)


def config_completa(config):
    return all(r in config for r in RANURAS)


def describir_config(config):
    ranuras = ", ".join(f"{r}={config[r]}" for r in RANURAS if r in config)
    extras = []
    if "temperatura" in config:
        extras.append(f"temp={config['temperatura']}")
    det = ", ".join(
        f"{r}:{palabras(CATALOGO[r][config[r]])}"
        for r in RANURAS
        if r in config
    )
    extra_txt = ", ".join(extras)
    if extra_txt:
        extra_txt = f", {extra_txt}"
    return f"{{{ranuras}{extra_txt}, partes=[{det or 'ninguna'}]}}"


def contar_configs_validas():
    return sum(
        1
        for c in CONFIGS
        if temp_valida(c["temperatura"])
        and all(r in c for r in RANURAS)
        and all(parte_valida(CATALOGO[r][c[r]]) for r in RANURAS)
    )


def es_valido(config):
    if "temperatura" in config and not temp_valida(config["temperatura"]):
        tipo = "completa" if config_completa(config) else "parcial"
        stats[f"podas_{tipo}"] += 1
        print(
            f"  PODA {tipo} (temperatura={config['temperatura']} <= {MIN_TEMP}): "
            f"{describir_config(config)}"
        )
        return False

    for r in RANURAS:
        if r not in config:
            continue
        t = CATALOGO[r][config[r]]
        if parte_valida(t):
            continue
        n = palabras(t)
        tipo = "completa" if config_completa(config) else "parcial"
        stats[f"podas_{tipo}"] += 1
        print(
            f"  PODA {tipo} ({r}={n} palabras, rango {MIN_PALABRAS}–{MAX_PALABRAS}): "
            f"{describir_config(config)}"
        )
        return False
    return True


def es_viable(config):
    global mejor
    costo = len(INSTANCIAS)
    restante = presupuesto_restante()

    if not puede_evaluar():
        stats["cortes_presupuesto"] += 1
        print(
            f"  CORTE presupuesto: no evalúo {describir_config(config)} "
            f"(gastado={oraculo.gastado}, restante={restante}, costo={costo})"
        )
        return False

    print(
        f"  EVALUAR {describir_config(config)} "
        f"→ hasta {costo} rollouts (gastado={oraculo.gastado}, restante={restante})"
    )
    antes = oraculo.gastado
    r = oraculo.evaluar(config, INSTANCIAS, semilla=1)
    delta = oraculo.gastado - antes
    stats["evaluadas"] += 1
    stats["rollouts_gastados"] += delta
    if delta == 0:
        stats["cache_hits"] += 1
    historial.append(r.precision)

    cache_txt = " cache-hit" if delta == 0 else ""
    mejor_txt = f"{mejor[0]:5.1%}" if mejor else "  n/a"
    if mejor is None or r.precision > mejor[0]:
        mejor = (r.precision, dict(config))
        stats["mejoras"] += 1
        print(
            f"    gastado {oraculo.gastado:4d} (+{delta})   esta {r.precision:5.1%}   "
            f"mejor {mejor[0]:5.1%}   ← nueva mejor{cache_txt}"
        )
        return True

    stats["sin_mejora"] += 1
    print(
        f"    gastado {oraculo.gastado:4d} (+{delta})   esta {r.precision:5.1%}   "
        f"mejor {mejor_txt}   (evaluada pero no mejora){cache_txt}"
    )
    return False


def backtracking(config, profundidad=0):
    indent = "  " * profundidad
    if not puede_evaluar():
        stats["cortes_presupuesto"] += 1
        print(f"{indent}CORTE presupuesto agotado en nodo {describir_config(config)}")
        return
    if not es_valido(config):
        return

    if config_completa(config):
        print(f"{indent}HOJA válida: {describir_config(config)}")
        es_viable(config)
        return

    siguiente = next(r for r in RANURAS if r not in config)
    print(f"{indent}RAMA {siguiente} desde {describir_config(config)}")
    for i in range(len(CATALOGO[siguiente])):
        config[siguiente] = i
        backtracking(config, profundidad + 1)
        del config[siguiente]
        if not puede_evaluar():
            return


In [ ]:
from oraculo import TEMPERATURAS, espacio

PRESUPUESTO = 100
INSTANCIAS = busqueda[:10]
COSTO_EVAL = len(INSTANCIAS)

CONFIGS = espacio(TEMPERATURAS)

mejor = None
historial = []
stats = {
    "podas_parcial": 0,
    "podas_completa": 0,
    "evaluadas": 0,
    "mejoras": 0,
    "sin_mejora": 0,
    "cortes_presupuesto": 0,
    "rollouts_gastados": 0,
    "cache_hits": 0,
}

oraculo.gastado = 0

validas_total = contar_configs_validas()
max_evals = PRESUPUESTO // COSTO_EVAL

print("=== ANTES DE BUSCAR ===")
print(f"presupuesto:        {PRESUPUESTO} rollouts")
print(f"instancias/eval:    {COSTO_EVAL}  →  cada evaluar cuesta hasta {COSTO_EVAL} rollouts")
print(f"máx. evaluaciones: {max_evals}  (= {PRESUPUESTO} / {COSTO_EVAL})")
print(f"espacio:            {len(CONFIGS)} configs  (temperaturas {TEMPERATURAS})")
print(
    f"configs válidas:    {validas_total} / {len(CONFIGS)}  "
    f"(temp > {MIN_TEMP}, cada ranura: {MIN_PALABRAS}–{MAX_PALABRAS} palabras, sin vacíos)"
)
if validas_total > max_evals:
    print(
        f"⚠ no alcanza el presupuesto para evaluar todas las válidas: "
        f"faltan {validas_total - max_evals} configs sin probar"
    )
print(
    "nota: es_viable evalúa TODA hoja válida aunque no mejore el score; "
    "‘viable’ solo decide si actualiza mejor."
)
print()

for temp in TEMPERATURAS:
    if not puede_evaluar():
        break
    print(f"--- temperatura {temp} ---")
    backtracking({"temperatura": temp})

print("\n=== RESUMEN ===")
print(f"rollouts gastados:  {stats['rollouts_gastados']} / {PRESUPUESTO}  (oraculo.gastado={oraculo.gastado})")
print(f"evaluaciones:       {stats['evaluadas']}  (cache-hits: {stats['cache_hits']})")
print(f"  mejoras:          {stats['mejoras']}")
print(f"  sin mejora:       {stats['sin_mejora']}  ← igual consumieron rollouts si delta > 0")
print(f"podas parciales:    {stats['podas_parcial']}  (temp ≤ {MIN_TEMP} o ranura fuera de rango)")
print(f"podas completas:    {stats['podas_completa']}  (temp ≤ {MIN_TEMP} o ranura fuera de rango)")
print(f"cortes presupuesto: {stats['cortes_presupuesto']}")
print(f"configs sin evaluar:{max(0, validas_total - stats['evaluadas'])}")

if mejor:
    print("\nmejor configuración:", mejor[1])
else:
    print("\nninguna configuración viable dentro del presupuesto")


### Leer los fallos — no cuesta nada


In [ ]:
r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Validar la config elegida  ·  no gasta presupuesto

Las familias de validación no se usaron al buscar. Sirve para ver si la config generaliza, no para elegir otra.

In [ ]:
from ayudas import validar

r_val = validar(oraculo, mejor[1], datos)
print("búsqueda (mejor):", f"{mejor[0]:.1%}")
print("validación:      ", f"{r_val.precision:.1%}")

## 5 · La entrega


In [ ]:
from ayudas import entrega
from google.colab import files

entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
